In [1]:
symbol = 'TQQQ'

import yfinance
hist = yfinance.download(tickers=symbol,period="7d",interval="1m")

[*********************100%***********************]  1 of 1 completed


In [2]:
# change column name to lower case and without space.
# option 1
new_col_list = []
for i in hist.columns:
    new_col_list.append(i.lower().replace(" ", ""))
hist.columns = new_col_list

In [3]:
# option 2 
# dataframe.columns.values == array
# dataframe.columns == index
# pandas does not want pd.Indexs to be mutable
for i,j in enumerate(hist.columns):
    hist.columns.values[i] = j.lower().replace(" ", "")


In [4]:
# in case zeros value in dataset, change zeros to ones for a better calculation
hist.replace(0,1,inplace=True)

In [5]:
import numpy as np
import pandas as pd
import pandas_datareader as pdr
import matplotlib.pyplot as plt

def get_bollinger_bands(data, rate=20, sd=2):
    sma = data['close'].rolling(rate).mean()
    std = data['close'].rolling(rate).std()
    bollinger_up = pd.Series(sma + std * sd, name = 'bollinger_up_' + str(rate)) # Calculate top band
    bollinger_down = pd.Series(sma - std * sd, name ='bollinger_down_' + str(rate)) # Calculate bottom band
    data = data.join([bollinger_up,bollinger_down])
    return data

# symbol = 'AAPL'
# df = pdr.DataReader(symbol, 'yahoo', '2014-07-01', '2015-07-01')
# df.index = np.arange(df.shape[0])




In [6]:
def SMA(data, ndays=20): 
    SMA = pd.Series(data['close'].rolling(ndays).mean(), name = 'SMA_'+ str(ndays)) 
    data = data.join(SMA) 
    return data

In [7]:
def EWMA(data, ndays=20): 
    EMA = pd.Series(data['close'].ewm(span = ndays, min_periods = ndays - 1).mean(), 
                 name = 'EWMA_' + str(ndays)) 
    data = data.join(EMA) 
    return data

In [8]:
def rsi(data, periods = 20):
    
    close_delta = data['close'].diff()

    # Make two series: one for lower closes and one for higher closes
    up = close_delta.clip(lower=0)
    down = -1 * close_delta.clip(upper=0)
    
    ma_up = up.ewm(com = periods - 1, adjust=True, min_periods = periods).mean()
    ma_down = down.ewm(com = periods - 1, adjust=True, min_periods = periods).mean()

    rsi = ma_up / ma_down
    rsi = 100 - (100/(1 + rsi))
    RSI = pd.Series(rsi, name='RSI_' + str(periods))
    data = data.join(RSI)
    return data

In [9]:
def gain(x):
    return ((x > 0) * x).sum()


def loss(x):
    return ((x < 0) * x).sum()


# Calculate money flow index
def mfi(data, n=20):
    high = data['high']
    low = data['low']
    close = data['close']
    volume = data['volume']
    typical_price = (high + low + close)/3
    money_flow = typical_price * volume
    mf_sign = np.where(typical_price > typical_price.shift(1), 1, -1)
    signed_mf = money_flow * mf_sign
    mf_avg_gain = signed_mf.rolling(n).apply(gain, raw=True)
    mf_avg_loss = signed_mf.rolling(n).apply(loss, raw=True)
    mfi = (100 - (100 / (1 + (mf_avg_gain / abs(mf_avg_loss)))))
    MFI = pd.Series(mfi, name='MFI_' + str(n))
    data = data.join(MFI)
    return data

In [10]:
def percent_change(data, level = 'close'):
    PER = pd.Series((data[level] - data[level].shift(1))/data[level].shift(1)*100, name = 'PER_'+level)
    data = data.join(PER)
    return data
    

In [11]:
list(range(7,28,7))

[7, 14, 21]

In [12]:
hist

,open,high,low,close,adjclose,volume
Datetime,,,,,,
2023-03-16 09:30:00-04:00,22.879999,22.980000,22.865000,22.879999,22.879999,11967424
2023-03-16 09:31:00-04:00,22.870001,22.930000,22.799999,22.809401,22.809401,1017352
2023-03-16 09:32:00-04:00,22.799999,22.861500,22.750000,22.790001,22.790001,761517
2023-03-16 09:33:00-04:00,22.790001,22.846201,22.770100,22.830000,22.830000,753542
2023-03-16 09:34:00-04:00,22.825001,22.868700,22.790001,22.860001,22.860001,690930
...,...,...,...,...,...,...
2023-03-24 15:55:00-04:00,25.610001,25.610001,25.549999,25.570000,25.570000,518317
2023-03-24 15:56:00-04:00,25.580000,25.680000,25.570000,25.670000,25.670000,580232
2023-03-24 15:57:00-04:00,25.663601,25.740000,25.660101,25.736700,25.736700,528545


In [17]:
length = range(7,28,7)
st = range(1,4,1)

final_table = []
for ii in length:
    for jj in st:
        sub = hist.copy()
    #     hist = mfi(hist,14)

    #     hist = rsi(hist, 14)

        sub = get_bollinger_bands(sub, ii,jj)

    #     hist = percent_change(hist, 'close')

    #     hist = percent_change(hist, 'volume')

        sub['under_BBL'] = np.where(sub['close'] < sub['bollinger_down_'+str(ii)], 1, 0)

        tb = [0,0]
        for i in np.unique(sub.index.date):
            print(i,ii,jj)
            daily_hist = sub[str(i):str(i)]

        #     plt.figure(figsize=(18,8))
        #     plt.title(symbol + ' Bollinger Bands')
        #     plt.xlabel('Days')
        #     plt.ylabel('Closing Prices')
        #     plt.plot(daily_hist['close'], label='Closing Prices')
        #     plt.plot(daily_hist['bollinger_up_14'], label='Bollinger Up', c='g')
        #     plt.plot(daily_hist['bollinger_down_14'], label='Bollinger Down', c='r')




            for i in range(5):
                daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
            daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
            daily_hist['success'] = np.where((daily_hist['under_BBL'] == 1) & (daily_hist['close']*1.001 < daily_hist['highest']), 1,0)

#             print(len(daily_hist[daily_hist['under_BBL']==1]), len(daily_hist[daily_hist['success']==1]), len(daily_hist[daily_hist['success']==1])/len(daily_hist[daily_hist['under_BBL']==1]))
        #     plt.scatter(daily_hist[daily_hist['success']==1]['close'].index, daily_hist[daily_hist['success']==1]['close'],label='success', marker = '^', color = 'r')
        #     plt.scatter(daily_hist[(daily_hist['success']==0) & (daily_hist['under_BBL']==1)]['close'].index, daily_hist[(daily_hist['success']==0) & (daily_hist['under_BBL']==1)]['close'],label='fail', marker = 'v', color = 'b')
        #     plt.legend()
        #     plt.show()



            tb[0] +=len(daily_hist[daily_hist['under_BBL']==1])
            tb[1] +=len(daily_hist[daily_hist['success']==1])
        final_table.append([ii,jj,tb[0],tb[1]])


2023-03-16 7 1
2023-03-17 7 1
2023-03-20 7 1
2023-03-21 7 1
2023-03-22 7 1
2023-03-23 7 1
2023-03-24 7 1
2023-03-16 7 2
2023-03-17 7 2
2023-03-20 7 2
2023-03-21 7 2
2023-03-22 7 2
2023-03-23 7 2
2023-03-24 7 2
2023-03-16 7 3
2023-03-17 7 3
2023-03-20 7 3
2023-03-21 7 3
2023-03-22 7 3
2023-03-23 7 3
2023-03-24 7 3


<ipython-input-17-b08f975a4b8a>:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['high_'+str(i)] = daily_hist['high'].shift(-1-i)
<ipython-input-17-b08f975a4b8a>:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  daily_hist['highest'] = daily_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
<ipython-input-17-b08f975a4b8a>:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See 

2023-03-16 14 1
2023-03-17 14 1
2023-03-20 14 1
2023-03-21 14 1
2023-03-22 14 1
2023-03-23 14 1
2023-03-24 14 1
2023-03-16 14 2
2023-03-17 14 2
2023-03-20 14 2
2023-03-21 14 2
2023-03-22 14 2
2023-03-23 14 2
2023-03-24 14 2
2023-03-16 14 3
2023-03-17 14 3
2023-03-20 14 3
2023-03-21 14 3
2023-03-22 14 3
2023-03-23 14 3
2023-03-24 14 3
2023-03-16 21 1
2023-03-17 21 1
2023-03-20 21 1
2023-03-21 21 1
2023-03-22 21 1
2023-03-23 21 1
2023-03-24 21 1
2023-03-16 21 2
2023-03-17 21 2
2023-03-20 21 2
2023-03-21 21 2
2023-03-22 21 2
2023-03-23 21 2
2023-03-24 21 2
2023-03-16 21 3
2023-03-17 21 3
2023-03-20 21 3
2023-03-21 21 3
2023-03-22 21 3
2023-03-23 21 3
2023-03-24 21 3


In [21]:
ttable = pd.DataFrame(final_table, columns = ['length','std','total','success'])

In [22]:
ttable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   length   9 non-null      int64
 1   std      9 non-null      int64
 2   total    9 non-null      int64
 3   success  9 non-null      int64
dtypes: int64(4)
memory usage: 416.0 bytes


In [24]:
ttable['s_rate'] = ttable['success']/ttable['total']

In [25]:
ttable

,length,std,total,success,s_rate
0,7,1,608,487,0.800987
1,7,2,22,14,0.636364
2,7,3,0,0,NaN
3,14,1,607,482,0.794069
4,14,2,106,88,0.830189
5,14,3,0,0,NaN
6,21,1,601,481,0.800333
7,21,2,120,96,0.800000
8,21,3,3,3,1.000000


In [28]:
ttable['expectation'] = 1.001**ttable['success']*0.999**(ttable['total']-ttable['success'])

In [29]:
ttable

,length,std,total,success,s_rate,expectation
0,7,1,608,487,0.800987,1.441517
1,7,2,22,14,0.636364,1.006007
2,7,3,0,0,NaN,1.000000
3,14,1,607,482,0.794069,1.428602
4,14,2,106,88,0.830189,1.072451
5,14,3,0,0,NaN,1.000000
6,21,1,601,481,0.800333,1.434333
7,21,2,120,96,0.800000,1.074591
8,21,3,3,3,1.000000,1.003003


In [27]:
# print('Success rate is ' + str(final_table['success'].sum()/final_table['total'].sum()*100))

In [ ]:
final_table['success'].sum(), final_table['total'].sum()

In [ ]:
# for i in np.unique(hist.index.date):
#     print(i)
#     sub_hist = hist[str(i):str(i)]

    
#     fig = plt.figure(figsize=(10, 7))

#     # Define position of 1st subplot
#     ax = fig.add_subplot(4, 1, 1)

#     # Set the title and axis labels
#     plt.title(symbol + ' Bollinger Bands')
#     plt.xlabel('Days')
#     plt.ylabel('Closing Prices')
#     plt.plot(sub_hist['close'], label='Closing Prices')
#     plt.plot(sub_hist['bollinger_up_14'], label='Bollinger Up')
#     plt.plot(sub_hist['bollinger_down_14'], label='Bollinger Down')
#     plt.legend()

#     # Define position of 2nd subplot
#     bx = fig.add_subplot(4, 1, 2)

#     # Set the title and axis labels
#     plt.title('Relative Strength Index')
#     plt.xlabel('Date')
#     plt.ylabel('RSI values')
    
#     plt.plot(sub_hist['MFI_14'], label='MFI_14')
#     plt.axhline(y=30, color='grey', linestyle='-')
#     plt.axhline(y=70, color='grey', linestyle='-')
#     plt.plot(sub_hist['RSI_14'], label='RSI_14') 

#     # Add a legend to the axis
#     plt.legend()

    
    
#     cx = fig.add_subplot(4, 1, 3)
#     plt.title('Pecentage Close Change')
#     plt.xlabel('Date')
#     plt.ylabel('Percentage values')
    
#     plt.plot(sub_hist['PER_close'], label='Percent close')

#     plt.legend()
    

#     cx = fig.add_subplot(4, 1, 4)
#     plt.title('Pecentage Volume Change')
#     plt.xlabel('Date')
#     plt.ylabel('Percentage values')
    
#     plt.plot(sub_hist['PER_volume'], label='Percent close')

#     plt.legend()
    
    
#     plt.tight_layout()
#     plt.show()
    
    
   

In [ ]:
# for i in np.unique(hist.index.date):
#     print(i)
#     sub_hist = hist[str(i):str(i)]

#     for i in range(5):
#         sub_hist['high_'+str(i)] = sub_hist['high'].shift(-1-i)
#     sub_hist['highest'] = sub_hist[["high_0", "high_1", "high_2", "high_3", "high_4"]].max(axis=1)
#     sub_hist['success'] = np.where(sub_hist['close']*1.001 < sub_hist['highest'], 1,0) # 0.1%
#     print(sub_hist)